In [17]:
# A empresa precisa responder:

# Qual foi o faturamento total considerando apenas ordens finalizadas?

# Quanto cada cliente faturou?

# Quanto cada cidade faturou?

# Quanto cada categoria de serviço faturou?

# Qual categoria teve maior faturamento?

# Qual foi o ticket médio por categoria?


In [18]:
# criando dfs
import pandas as pd

clientes = pd.read_csv('../data/clientes.csv')

ordens = pd.read_csv('../data/ordens_servico.csv')

servicos = pd.read_csv('../data/servicos.csv')

display(clientes.head(3), ordens.head(3), servicos.head(3))

,id_cliente,nome_cliente,cidade
0,1,João,São Paulo
1,2,Maria,São Paulo
2,3,Ana,Guarulhos


,id_os,id_cliente,id_servico,valor,status,data_os
0,1,1,1,120,Finalizado,2026-01-05
1,2,2,2,80,Finalizado,2026-01-05
2,3,1,3,350,Finalizado,2026-01-06


,id_servico,nome_servico,categoria
0,1,Troca de Óleo,Manutenção
1,2,Alinhamento,Serviços Rápidos
2,3,Freio,Segurança


In [19]:
# unindo tabelas
# princiapal: ordens

ordens_cliente = ordens.merge(clientes, how='left', on='id_cliente')
ordens_cliente_servico = ordens_cliente.merge(servicos, how='left', on='id_servico')
df = ordens_cliente_servico
display(df.head(2))

,id_os,id_cliente,id_servico,valor,status,data_os,nome_cliente,cidade,nome_servico,categoria
0,1,1,1,120,Finalizado,2026-01-05,João,São Paulo,Troca de Óleo,Manutenção
1,2,2,2,80,Finalizado,2026-01-05,Maria,São Paulo,Alinhamento,Serviços Rápidos


In [20]:
# ordenando colunas
df = df[[
    'id_cliente',
    'nome_cliente',
    'cidade',
    'id_os',
    'id_servico',
    'categoria',
    'nome_servico',
    'valor',
    'data_os',
    'status'
]]

display(df.head(4))

,id_cliente,nome_cliente,cidade,id_os,id_servico,categoria,nome_servico,valor,data_os,status
0,1,João,São Paulo,1,1,Manutenção,Troca de Óleo,120,2026-01-05,Finalizado
1,2,Maria,São Paulo,2,2,Serviços Rápidos,Alinhamento,80,2026-01-05,Finalizado
2,1,João,São Paulo,3,3,Segurança,Freio,350,2026-01-06,Finalizado
3,4,Carlos,Suzano,4,1,Manutenção,Troca de Óleo,120,2026-01-06,Pendente


In [21]:
# Qual foi o faturamento total considerando apenas ordens finalizadas?
df = df[df['status'] == 'Finalizado']

faturamento_total = df['valor'].sum()

In [22]:
# Quanto cada cliente faturou?

faturamento_cliente = df.groupby('id_cliente')['valor'].sum()

In [23]:
# Quanto cada cidade faturou?
faturamento_cidade = df.groupby('cidade')['valor'].sum()

In [24]:
# Quanto cada categoria de serviço faturou?
faturamento_categoria = df.groupby('categoria')['valor'].sum()

In [25]:
# Qual categoria teve maior faturamento?
melhor_categoria = faturamento_categoria.idxmax()

In [26]:
# Qual foi o ticket médio por categoria?
media_categoria = df.groupby('categoria')['valor'].mean().round(1)

In [27]:
# Quantas ordens finalizadas existem por categoria?
os_categoria = df.groupby('categoria')['id_os'].count()

In [28]:
# criando relatorio_clientes.csv

relatorio_clientes = pd.DataFrame({
    'id_cliente': df.groupby('id_cliente')['id_os'].count().index,
    'nome_cliente': df.groupby('id_cliente')['nome_cliente'].first(),
    'cidade': df.groupby('id_cliente')['cidade'].first(),
    'faturamento_total': df.groupby('id_cliente')['valor'].sum(),
    'qtd_ordens': df.groupby('id_cliente')['id_os'].count(),
    'ticket_medio': df.groupby('id_cliente')['valor'].mean()
})
display(relatorio_clientes)

#exportando relatorio
relatorio_clientes.to_csv('../output/relatorio_clientes.csv', index=False)

,id_cliente,nome_cliente,cidade,faturamento_total,qtd_ordens,ticket_medio
id_cliente,,,,,,
1,1,João,São Paulo,1440,4,360.0
2,2,Maria,São Paulo,1010,4,252.5
3,3,Ana,Guarulhos,590,2,295.0
4,4,Carlos,Suzano,80,1,80.0
5,5,Pedro,Mogi das Cruzes,450,1,450.0
6,6,Juliana,Guarulhos,1050,3,350.0


In [29]:
# criando relatorio_categorias.csv
relatorio_categorias = pd.DataFrame({
    'categoria': faturamento_categoria.index,
    'faturamento_total': faturamento_categoria,
    'qtd_ordens': os_categoria,
    'ticket_medio': media_categoria,
    'participacao_percentual': (faturamento_categoria/faturamento_total*100).round(1)
})

relatorio_categorias.to_csv('../output/relatorio_categorias.csv', index=False)

In [30]:
# criando relatorio_cidades.csv
relatorio_cidades = pd.DataFrame({
    'cidade': faturamento_cidade.index,
    'faturamento_total': faturamento_cidade,
    'qtd_ordens': df.groupby('cidade')['id_os'].count(),
    'ticket_medio': df.groupby('cidade')['valor'].mean().round(1),
    'participacao_percentual': (faturamento_cidade/faturamento_total*100).round(1)
})

relatorio_cidades.to_csv('../output/relatorio_cidades.csv', index=False)